In [4]:
import numpy as np
import xarray as xr
import toml
import munch
from tqdm import tqdm
import torch
import datetime
import zarr

import warnings
warnings.filterwarnings("ignore")

import os
os.environ['RUCIO_CONFIG'] = '/home/jovyan/work/ML4Fires/rucio.cfg'

from typing import Any
from rucio.client.client import Client
from types import SimpleNamespace
#from rucio.client.uploadclient import UploadClient
rucio = Client()

from Fires._utilities.utils_mlflow import load_model_from_mlflow
from Fires._utilities.utils_inference import get_cmip6_inference,load_cmip6_files_from_config, get_cmip6_files_rucio,_get_file_list

In [ ]:
variable = "*"
frequency = "*"
model = "CMCC-ESM2"
scenario = "*"
filters = variable + "_*_" + model + "_" + scenario + "_*.nc"
listing = rucio.list_dids(scope="abennasser", filters={"name": filters}, did_type="file")
#filelist = list(listing)
import pprint
#pprint.pprint(len(sorted(filelist)))
rse="VEGA-DCACHE"
dids = []
for l in listing:
    dids.append({"scope": "abennasser", "name": l})
replicas = rucio.list_replicas(
    dids=dids,
    schemes=["file"],
    rse_expression=rse
)
datapath=[]
for r in replicas:
        if rse in r["rses"]:
            lfilepath=r["rses"][rse][0]
            filepath = lfilepath.replace('file://localhost', '')
            datapath.append(filepath)


In [6]:
from IPython.display import display, clear_output
import ipywidgets as widgets

# -----------------------------
# UI widgets
# -----------------------------
scenario = widgets.Dropdown(
    options=[
        ('SSP126', 'ssp126'),
        ('SSP245', 'ssp245'),
        ('SSP370', 'ssp370'),
        ('SSP585', 'ssp585')
    ],
    value='ssp126',
    description='CMIP6 Scenario:',
    style={'description_width': '150px'}
)

year_range = widgets.IntRangeSlider(
    value=[2030, 2035],
    min=2015,
    max=2100,
    step=1,
    description='Year Range:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

run_button = widgets.Button(
    description='Load CMIP6 Data',
    button_style='success',
    icon='cloud-download'
)

output = widgets.Output()

# -----------------------------
# Callback using your final function
# -----------------------------
def on_run_button_clicked(b):
    with output:
        clear_output()
        print(f"📌 Scenario selected: {scenario.value}")
        print(f"📆 Year range selected: {year_range.value}")
        
        try:
            files_dict, date_windows = load_cmip6_files_from_config(
                scenario=scenario.value,
                year_range=tuple(year_range.value)
            )
            print("✅ Files loaded successfully.")
            for var, files in files_dict.items():
                print(f"🔹 {var}: {len(files)} file(s)")
        except Exception as e:
            print(f"❌ Error loading files:\n{e}")

# Hook up the callback
run_button.on_click(on_run_button_clicked)

# -----------------------------
# Display everything
# -----------------------------
display(widgets.VBox([
    widgets.HBox([scenario, year_range]),
    run_button,
    output
]))
